[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Losses and Optimizers


## What you will be able to do

Choose the loss that fits a question, give it outputs and answers of the shapes and types it
expects, and choose an optimizer and a learning rate that train a model in a reasonable number of
steps.

## The idea

### The problem

Training moves a model's weights in whatever direction lowers one number, the loss. For the January
mean, a temperature, the mean squared error was the obvious choice. Whether a station has snow on 1
May is a yes-or-no question, and a model that gives a snowy station a probability of 0.7 is less
wrong than one that gives it 0.2, so its loss has to measure probabilities. Sorting the stations
into three temperature bands needs a loss that compares three scores with one right answer.

Each loss expects its inputs in one shape and one dtype. Given a target of the wrong shape, PyTorch's
mean squared error warns and carries on, and the model it trains learns nothing but the average. The other half of training is the optimizer, which turns gradients into a step, and its
learning rate sets how far each step goes.

### A formal definition

> A **loss function** takes a model's outputs for a batch of examples and the right answers, and
> returns one number that training lowers. A model's raw outputs, before a sigmoid or a softmax
> turns them into probabilities, are its **logits**. An **optimizer** holds a model's parameters
> and, each time its `step()` is called, moves them using their gradients by a rule such as plain
> gradient descent, momentum or Adam.

### Why it works that way

The classification losses take logits because a probability computed first loses precision. In
32-bit floats, the sigmoid of 40 is exactly 1.0, and the log of 1 − 1.0 is minus infinity. A loss
that starts from the logit computes the same quantity without ever forming that 1.0. For several
classes, `CrossEntropyLoss` applies the softmax itself. It takes the right answer as a class number,
an integer, or as a row of floats with a column per class, and Keras has a separate loss for each of
the two forms.

Optimizers differ in how far they move each weight. Plain gradient descent moves a weight by the
learning rate times its gradient, so weights with large gradients move far and the rest barely move.
Momentum adds a share of the previous step to each new one, so steps that keep pointing the same way
build up speed. Adam divides each weight's step by a running measure of that weight's gradient, so
at first every weight with a gradient moves by about the learning rate. The same learning
rate therefore means something different to the two.

A PyTorch loss takes the prediction first and the answer second, `loss_fn(output, target)`, and a
Keras loss the reverse, `loss_fn(y_true, y_pred)`.

### Where this shows up

The **Training Loops** notebook trains the snow classifier with this notebook's loss, and the
**Convolutional Networks** notebook classifies images with `CrossEntropyLoss`. Weight decay, in the
**Overfitting and Regularization** notebook, is a setting of the optimizer.

### What this notebook covers

- mean squared error in both frameworks, and the order of the arguments
- logits, probabilities, and why the losses want logits
- a snow classifier with `BCEWithLogitsLoss`, and three temperature bands with `CrossEntropyLoss`
- what one optimizer step does to the weights
- plain gradient descent, momentum and Adam side by side
- four mistakes with shapes, dtypes and sigmoids

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

logits = torch.tensor([[-2.0], [0.0], [3.0]])
labels = torch.tensor([[0.0], [1.0], [1.0]])
print(torch.sigmoid(logits).ravel())
print(torch.nn.BCEWithLogitsLoss()(logits, labels))
```

```
tensor([0.1192, 0.5000, 0.9526])
tensor(0.2896)
```

The sigmoid turns each logit into a probability, and a logit of 0 is a probability of exactly 0.5.
The loss takes the logits themselves with the right answers, and it is smallest when every example
with a 1 gets a large logit and every example with a 0 a very negative one.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, a way to catch warnings, and the stations.

- `numpy` builds the stations and the temperature bands
- `torch` is PyTorch, whose losses are in `torch.nn` and optimizers in `torch.optim`
- `tensorflow` is TensorFlow, and `keras` its library of layers, losses and optimizers
- `contextlib` and `os` hide the messages TensorFlow prints while it loads
- `warnings` catches two warnings in Common errors, so that each prints as one line

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
TensorFlow logs once it has started, and `c_messages_hidden` sends the lines it prints while it
starts to `os.devnull`. The cell prints PyTorch's version without the part after `+`, which names
the build.

`station_network_arrays` makes the 1,200 made-up weather stations of the **scikit-learn, Deep Dive**
guide, drawn from a fixed seed, so they are the same on every run. `features` holds each station's
latitude, longitude, elevation and distance from the coast as 32-bit floats, `jan_mean` its January
mean in degrees Celsius, and `snow_on_may_1` a 1 for a station with snow on 1 May and a 0 otherwise.
The cell splits the stations into 800 for training, 200 for validation and 200 for testing, and
standardizes the features with the training stations' means and standard deviations into `scaled`.

In [1]:
import contextlib
import os
import warnings

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### The training and test stations as tensors

Every target gets a column of its own, shape (800, 1), to match a model that makes one prediction
per station in a column.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
X_test = torch.from_numpy(scaled[test_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
snow_train = torch.from_numpy(snow_on_may_1[train_rows]).unsqueeze(1)
snow_test = torch.from_numpy(snow_on_may_1[test_rows]).unsqueeze(1)

print(X_train.shape, y_train.shape, snow_train.shape)
print("share with snow:", round(snow_train.mean().item(), 3), "training,", round(snow_test.mean().item(), 3), "test")

torch.Size([800, 4]) torch.Size([800, 1]) torch.Size([800, 1])
share with snow: 0.374 training, 0.465 test


The test stations happen to have more snow than the training stations, 0.465 against 0.374, which a
random split of 1,200 stations can give.

### Mean squared error by hand and in both frameworks

The cell compares three predictions of the January mean with the temperatures that were measured.

In [3]:
predicted = torch.tensor([[-3.0], [-7.5], [0.5]])
measured = torch.tensor([[-2.0], [-8.0], [1.5]])

print("by hand:", ((predicted - measured) ** 2).mean().item())
print("PyTorch:", torch.nn.MSELoss()(predicted, measured).item())
print("Keras:  ", float(keras.losses.MeanSquaredError()(measured.numpy(), predicted.numpy())))

by hand: 0.75
PyTorch: 0.75
Keras:   0.75


The errors are 1, 0.5 and 1 degree, whose squares average 0.75. PyTorch's loss is called with the
prediction first, and Keras's with the measured value first, and mean squared error gives the same
answer with its arguments in either order.

### Logits and probabilities

`torch.sigmoid` turns a logit into a probability. `BCEWithLogitsLoss` takes logits, and `BCELoss`
takes probabilities, so given the same examples the two should agree.

In [4]:
logits = torch.tensor([[-2.0], [0.0], [3.0]])
labels = torch.tensor([[0.0], [1.0], [1.0]])

print("from logits:       ", round(torch.nn.BCEWithLogitsLoss()(logits, labels).item(), 4))
print("from probabilities:", round(torch.nn.BCELoss()(torch.sigmoid(logits), labels).item(), 4))

from logits:        0.2896
from probabilities: 0.2896


The two agree. The next cell gives a logit of 40, from a model very sure of snow, to an example whose
answer is 0.

In [5]:
sure = torch.tensor([[40.0]])
wrong = torch.tensor([[0.0]])

print("probability:       ", torch.sigmoid(sure).item())
print("from probabilities:", torch.nn.BCELoss()(torch.sigmoid(sure), wrong).item())
print("from logits:       ", torch.nn.BCEWithLogitsLoss()(sure, wrong).item())

probability:        1.0
from probabilities: 100.0
from logits:        40.0


In 32-bit floats the sigmoid of 40 is exactly 1.0, so `BCELoss` has to take the log of 0. It
clamps that log at −100 and reports a loss of 100. The true loss is 40, which is what
`BCEWithLogitsLoss` returns, since it never forms the probability, and that is why a classification
loss is given the logits.

### A snow classifier trained on logits

A linear model of four features with one output is a logistic regression when its output is read
as a logit. It predicts snow wherever the logit is above 0, which is a probability above 0.5.

In [6]:
torch.manual_seed(0)
snow_model = torch.nn.Linear(4, 1)
optimizer = torch.optim.SGD(snow_model.parameters(), lr=0.1)
bce = torch.nn.BCEWithLogitsLoss()

for step in range(500):
    optimizer.zero_grad()
    loss = bce(snow_model(X_train), snow_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_logits = snow_model(X_test)
accuracy = ((test_logits > 0).float() == snow_test).float().mean().item()
print("training loss", round(loss.item(), 3), "| test accuracy", round(accuracy, 3))
print("always no snow:", round(1 - snow_test.mean().item(), 3))
print("first three probabilities:", torch.sigmoid(test_logits[:3]).ravel().numpy().round(3))

training loss 0.329 | test accuracy 0.8
always no snow: 0.535
first three probabilities: [0.299 0.51  0.064]


The model is right about 80% of the test stations, where always answering no would be right about
53.5% of them. The first three test stations get probabilities of snow of 0.299, 0.51 and 0.064.

Keras takes the same choice as an argument. `BinaryCrossentropy(from_logits=True)` reads the
outputs as logits, and `BinaryAccuracy(threshold=0.0)` counts a logit above 0 as a prediction of
snow, since its default threshold of 0.5 is meant for probabilities.

In [7]:
keras.utils.set_random_seed(0)
snow_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(1)])
snow_keras.compile(optimizer=keras.optimizers.SGD(0.1),
                   loss=keras.losses.BinaryCrossentropy(from_logits=True),
                   metrics=[keras.metrics.BinaryAccuracy(threshold=0.0)])
history = snow_keras.fit(scaled[train_rows], snow_on_may_1[train_rows][:, None],
                         epochs=500, batch_size=800, verbose=0)

test_loss, test_accuracy = snow_keras.evaluate(scaled[test_rows], snow_on_may_1[test_rows][:, None], verbose=0)
print("training loss", round(history.history["loss"][-1], 3), "| test accuracy", round(test_accuracy, 3))

training loss 0.328 | test accuracy 0.81


Keras's model reaches a training loss of 0.328 against PyTorch's 0.329, and 0.81 on the test
stations, two stations more out of 200 than PyTorch's 0.8. The two started from different random
weights.

### Temperature bands with CrossEntropyLoss

`np.digitize` sorts the January means into bands below −8 degrees, from −8 up to −3, and −3 and
above, numbered 0, 1 and 2. A model for three classes has three outputs, one logit per band.
`CrossEntropyLoss` takes those logits and the band number of each station, as 64-bit integers.

In [8]:
BANDS = ["below -8", "-8 to -3", "-3 and above"]
band = np.digitize(jan_mean, [-8.0, -3.0])
band_train = torch.from_numpy(band[train_rows])
band_test = torch.from_numpy(band[test_rows])
print(band_train.dtype, band_train[:6])
print("stations per band:", dict(zip(BANDS, np.bincount(band).tolist())))

torch.manual_seed(0)
band_model = torch.nn.Linear(4, 3)
optimizer = torch.optim.SGD(band_model.parameters(), lr=0.5)
cross_entropy = torch.nn.CrossEntropyLoss()
for step in range(500):
    optimizer.zero_grad()
    loss = cross_entropy(band_model(X_train), band_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    band_logits = band_model(X_test)
print("training loss", round(loss.item(), 3), "| test accuracy",
      round((band_logits.argmax(dim=1) == band_test).float().mean().item(), 3))

torch.int64 tensor([0, 1, 1, 2, 1, 2])
stations per band: {'below -8': 198, '-8 to -3': 704, '-3 and above': 298}
training loss 0.311 | test accuracy 0.865


The model puts 86.5% of the test stations in the right band. `argmax(dim=1)` picks the band with the
largest logit for each station. For probabilities, `torch.softmax` turns each station's three logits
into three numbers that add up to 1.

In [9]:
print(band_logits[:2].numpy().round(2))
print(torch.softmax(band_logits[:2], dim=1).numpy().round(3))

[[-1.95  2.54 -1.05]
 [ 0.58  3.46 -5.88]]
[[0.011 0.963 0.027]
 [0.053 0.947 0.   ]]


The first station's logits favor the middle band, and the softmax gives that band a probability of
0.963. Keras calls the same loss `SparseCategoricalCrossentropy`, the sparse form taking a band
number per station.

In [10]:
keras.utils.set_random_seed(0)
band_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(3)])
band_keras.compile(optimizer=keras.optimizers.SGD(0.5),
                   loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                   metrics=["accuracy"])
history = band_keras.fit(scaled[train_rows], band[train_rows], epochs=500, batch_size=800, verbose=0)

test_loss, test_accuracy = band_keras.evaluate(scaled[test_rows], band[test_rows], verbose=0)
print("training loss", round(history.history["loss"][-1], 3), "| test accuracy", round(test_accuracy, 3))

training loss 0.312 | test accuracy 0.865


The Keras model puts the same share of the test stations in the right band, 0.865.

### What one optimizer step does

An optimizer is given the model's parameters when it is made, and from then on `step()` moves them
using whatever is in their `.grad`. For plain gradient descent, a step is the weight minus the
learning rate times the gradient, which the cell checks against what `step()` did.

In [11]:
torch.manual_seed(0)
linear = torch.nn.Linear(4, 1)
optimizer = torch.optim.SGD(linear.parameters(), lr=0.1)
print("gradient before backward:", linear.weight.grad)

before = linear.weight.detach().clone()
torch.nn.MSELoss()(linear(X_train), y_train).backward()
expected = before - 0.1 * linear.weight.grad
optimizer.step()

print("weights before:", before.numpy().round(3))
print("gradient:      ", linear.weight.grad.numpy().round(3))
print("weights after: ", linear.weight.detach().numpy().round(3))
print("the same as weights − 0.1 × gradient, to rounding:",
      torch.allclose(linear.weight.detach(), expected, rtol=0, atol=1e-6))

optimizer.zero_grad()
print("gradient after zero_grad:", linear.weight.grad)

gradient before backward: None
weights before: [[-0.004  0.268 -0.412 -0.368]]
gradient:       [[4.514 4.441 1.975 1.912]]
weights after:  [[-0.455 -0.176 -0.609 -0.559]]
the same as weights − 0.1 × gradient, to rounding: True
gradient after zero_grad: None


Before `backward()` the gradient is `None`, and `step()` moved each weight by 0.1 times its
gradient. `zero_grad()` sets each `.grad` back to `None`, which the next `backward()` fills again.

### Momentum and Adam against plain gradient descent

`losses_by_step` trains the same small network as in the **Why PyTorch and TensorFlow** notebook,
from the same seed, with the optimizer it is given, and returns the training loss at four points.

In [12]:
def losses_by_step(make_optimizer, steps=(10, 50, 100, 200)):
    """Train a fresh 4-32-1 network on the January means and return its loss at each of steps."""
    torch.manual_seed(0)
    net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
    optimizer = make_optimizer(net.parameters())
    losses = []
    for step in range(1, max(steps) + 1):
        optimizer.zero_grad()
        loss = torch.nn.MSELoss()(net(X_train), y_train)
        loss.backward()
        optimizer.step()
        if step in steps:
            losses.append(round(loss.item(), 3))
    return losses


print("                    step 10   50  100  200")
print("SGD, 0.01          ", losses_by_step(lambda p: torch.optim.SGD(p, lr=0.01)))
print("SGD, 0.1           ", losses_by_step(lambda p: torch.optim.SGD(p, lr=0.1)))
print("SGD, 0.01, momentum", losses_by_step(lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9)))
print("Adam, 0.001        ", losses_by_step(lambda p: torch.optim.Adam(p, lr=0.001)))
print("Adam, 0.01         ", losses_by_step(lambda p: torch.optim.Adam(p, lr=0.01)))
print("Adam, 0.1          ", losses_by_step(lambda p: torch.optim.Adam(p, lr=0.1)))

                    step 10   50  100  200
SGD, 0.01           [9.464, 1.701, 1.31, 1.03]
SGD, 0.1            [2.02, 1.626, 1.196, 0.919]
SGD, 0.01, momentum [9.522, 0.972, 0.79, 0.764]
Adam, 0.001         [33.65, 28.844, 22.518, 8.845]
Adam, 0.01          [24.365, 2.608, 1.256, 0.89]
Adam, 0.1           [3.26, 0.86, 0.781, 0.751]


At a learning rate of 0.01, plain gradient descent is ahead of Adam after 10 steps, 9.464 against
24.365, and behind it after 200, 1.03 against 0.89. Momentum at the same learning rate reaches
0.764, Adam at 0.1 reaches 0.751, and Adam at 0.001, at 8.845, has come down only to about the loss
of predicting the average January mean for every station. These are one network from one seed, so
the table ranks nothing in general, and within it the learning rate changes the loss as much as the
choice of optimizer does. The first step of each optimizer at 0.01 shows why plain gradient descent
starts faster.

In [13]:
for name, make in [("SGD, 0.01", lambda p: torch.optim.SGD(p, lr=0.01)),
                   ("Adam, 0.01", lambda p: torch.optim.Adam(p, lr=0.01))]:
    torch.manual_seed(0)
    first_net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
    optimizer = make(first_net.parameters())
    before = torch.cat([p.detach().ravel().clone() for p in first_net.parameters()])
    torch.nn.MSELoss()(first_net(X_train), y_train).backward()
    optimizer.step()
    moved = (torch.cat([p.detach().ravel() for p in first_net.parameters()]) - before).abs()
    print(name, "| median move", round(moved.median().item(), 4), "| largest", round(moved.max().item(), 4))

SGD, 0.01 | median move 0.002 | largest 0.0988
Adam, 0.01 | median move 0.01 | largest 0.01


Adam's first step moves every weight that has a gradient by 0.01. Plain gradient descent at 0.01
moves most weights by less than that, but it moves the few with the largest gradients, among them
the output's bias, almost ten times as far, and while every prediction is far from the January
means those few are the ones that lower the loss.

### Keras's optimizers in a loop of your own

A Keras optimizer can drive a loop written with `GradientTape`, as `fit` does underneath.
`apply_gradients` takes pairs of a gradient and the variable it belongs to, and
`trainable_variables` lists a model's weights in the same order as the gradients the tape returns.

In [14]:
keras.utils.set_random_seed(0)
linear_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(1)])
optimizer = keras.optimizers.SGD(0.1)
mse = keras.losses.MeanSquaredError()
X_train_tf = tf.constant(scaled[train_rows])
y_train_tf = tf.constant(jan_mean[train_rows][:, None])

for step in range(500):
    with tf.GradientTape() as tape:
        loss = mse(y_train_tf, linear_keras(X_train_tf))
    gradients = tape.gradient(loss, linear_keras.trainable_variables)
    optimizer.apply_gradients(zip(gradients, linear_keras.trainable_variables))

print("training loss", round(float(loss), 3))

training loss 0.992


The training loss is 0.992, the loss the same linear model reached in the **Why PyTorch and
TensorFlow** notebook with PyTorch's optimizer and with Keras's `fit`.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/08-losses-and-optimizers-solutions.ipynb).

**1.** Compute the mean absolute error of `predicted` and `measured` with `torch.nn.L1Loss` and with `keras.losses.MeanAbsoluteError`, and check both against the mean of the absolute differences.

In [15]:
# your code here

**2.** Find the logit whose probability is 0.9 with `torch.logit`, and check it with `torch.sigmoid`.

In [16]:
# your code here

**3.** Train the snow classifier with `torch.optim.Adam` at a learning rate of 0.01 for 500 steps, from `torch.manual_seed(0)`, and print its accuracy on the test stations.

In [17]:
# your code here

**4.** Use `losses_by_step` to try plain gradient descent at a learning rate of 0.1 with `momentum=0.9`, and compare its losses with the table's.

In [18]:
# your code here

**5.** Train the three-band classifier in Keras with `keras.losses.CategoricalCrossentropy(from_logits=True)`, which takes a column per band, with labels from `keras.utils.to_categorical(band, 3)`, and print its accuracy on the test stations.

In [19]:
# your code here

## Common errors

### `UserWarning: Using a target size (torch.Size([800])) that is different to the input size (torch.Size([800, 1])).`

The cell trains the linear model of the January mean again, against `jan_mean` made into a tensor as
it is, with no `unsqueeze(1)`.

In [20]:
flat_target = torch.from_numpy(jan_mean[train_rows])
torch.manual_seed(0)
linear = torch.nn.Linear(4, 1)
optimizer = torch.optim.SGD(linear.parameters(), lr=0.1)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    for step in range(500):
        optimizer.zero_grad()
        loss = torch.nn.MSELoss()(linear(X_train), flat_target)
        loss.backward()
        optimizer.step()

print(f"{caught[0].category.__name__}: {str(caught[0].message).split('. ')[0]}.")
print(len(caught), "warnings, one per step")
print("training loss", round(loss.item(), 3), "| weights", linear.weight.detach().numpy().round(3),
      "| bias", round(linear.bias.item(), 3))
print("mean of the January means:", round(float(jan_mean[train_rows].mean()), 3),
      "| their variance:", round(float(jan_mean[train_rows].var()), 3))

500 warnings, one per step
training loss 8.481 | weights [[-0.005  0.005 -0.     0.   ]] | bias -5.108
mean of the January means: -5.108 | their variance: 8.481


The model's output has shape (800, 1) and the target (800,), and subtracting one from the other
broadcasts to (800, 800), which compares every station's prediction with every station's
temperature. The prediction that is closest on average to all 800 temperatures is their mean, so
training drives the four weights to about 0 and drives the bias to the mean. PyTorch issues the
warning at every step, 500 times here, which the cell counts by setting warnings to `"always"`. The
loss stops at the variance of the January means, where the same model trained with `unsqueeze(1)`
reaches 0.992, so give the target the output's shape.

In [21]:
torch.manual_seed(0)
linear = torch.nn.Linear(4, 1)
optimizer = torch.optim.SGD(linear.parameters(), lr=0.1)
for step in range(500):
    optimizer.zero_grad()
    loss = torch.nn.MSELoss()(linear(X_train), flat_target.unsqueeze(1))
    loss.backward()
    optimizer.step()
print("training loss", round(loss.item(), 3))

training loss 0.992


### `ValueError: Target size (torch.Size([800])) must be the same as input size (torch.Size([800, 1]))`

`BCEWithLogitsLoss` checks the same mismatch and refuses it.

In [22]:
torch.nn.BCEWithLogitsLoss()(snow_model(X_train), torch.from_numpy(snow_on_may_1[train_rows]))

ValueError: Target size (torch.Size([800])) must be the same as input size (torch.Size([800, 1]))

The fix is the same `unsqueeze(1)` on the target, which `snow_train` already has.

### `RuntimeError: expected scalar type Long but found Float`

Here the band numbers are stored as floats.

In [23]:
cross_entropy(band_model(X_train), band_train.float())

RuntimeError: expected scalar type Long but found Float

`CrossEntropyLoss` given one number per example reads it as a class number, which must be an
integer. `Long` is PyTorch's name for a 64-bit integer, and `Float` for a 32-bit float. Labels read
from a file or built with NumPy's floats arrive as floats. Convert them with `.long()`.

In [24]:
print(round(cross_entropy(band_model(X_train), band_train.float().long()).item(), 3))

0.311


The loss is back to 0.311, the band model's training loss from the worked examples.

### No error, and a sigmoid applied twice

This model ends with its own `nn.Sigmoid` and is trained with `BCEWithLogitsLoss`, which applies a
sigmoid as well.

In [25]:
torch.manual_seed(0)
squashed = torch.nn.Sequential(torch.nn.Linear(4, 1), torch.nn.Sigmoid())
optimizer = torch.optim.SGD(squashed.parameters(), lr=0.1)
for step in range(500):
    optimizer.zero_grad()
    loss = bce(squashed(X_train), snow_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    probabilities = torch.sigmoid(squashed(X_train))
accuracy = ((squashed(X_test) > 0.5).float() == snow_test).float().mean().item()
print("training loss", round(loss.item(), 3), "| test accuracy", round(accuracy, 3))
print("smallest and largest probability the loss saw:",
      round(probabilities.min().item(), 3), round(probabilities.max().item(), 3))

training loss 0.65 | test accuracy 0.77
smallest and largest probability the loss saw: 0.501 0.731


The model's outputs are already between 0 and 1, and the loss reads them as logits, so the
probability it scores is the sigmoid of a number between 0 and 1, which lies between 0.5 and 0.731.
Across the training stations those probabilities run from 0.501 to 0.731. The loss can never see a
station given a low chance of snow, and the model trains to a loss of 0.65 and a test accuracy of
0.77, against 0.329 and 0.8 for the same model without its sigmoid. Keras warns when it is given the
same mistake.

In [26]:
keras.utils.set_random_seed(0)
sigmoid_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(1, activation="sigmoid")])
dense = sigmoid_keras.layers[0]
behind = scaled[train_rows] @ dense.kernel.numpy() + dense.bias.numpy()
bce_keras = keras.losses.BinaryCrossentropy(from_logits=True)
targets = snow_on_may_1[train_rows][:, None]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    from_output = float(bce_keras(targets, sigmoid_keras(scaled[train_rows])))
    from_logits = float(bce_keras(targets, behind))
print(f"{caught[0].category.__name__}: {str(caught[0].message).split(' Was this')[0]}")
print("from the sigmoid's output:", round(from_output, 3), "| from the logits behind it:", round(from_logits, 3))

from the sigmoid's output: 0.918 | from the logits behind it: 0.918


Both losses are 0.918. Keras keeps track of where its sigmoid's output came from, warns, and computes
the loss from the logits behind it, which `behind` recomputes from the layer's weights. PyTorch has
no such check. Keep the sigmoid out of a model trained with `BCEWithLogitsLoss`, and apply
`torch.sigmoid` to its outputs only when you want probabilities.

## Recap

- You can choose mean squared error, binary cross-entropy or cross-entropy for a number, a yes or no,
  or one of several classes, in both frameworks.
- You can give a classification loss logits, and turn logits into probabilities with a sigmoid or a
  softmax when you want them.
- You can shape a target to match a model's output, and store class numbers as integers.
- You can make an optimizer, step it, and compare plain gradient descent, momentum and Adam at a few
  learning rates.

## What is next

The **Batches and Data Pipelines** notebook feeds a model a few examples at a time, with PyTorch's
`DataLoader` and TensorFlow's `tf.data`, instead of every training station at every step.

---

&#8592; **Previous:** [Reading Shape Errors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/07-reading-shape-errors.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Batches and Data Pipelines](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/09-batches-and-data-pipelines.ipynb) &#8594;
